# Legal Expert System — GPT-2 Fine-Tuning

Fine-tunes GPT-2 on Indian legal case data to generate judgement justifications for IPC/BNS sections.

| | |
|---|---|
| **Dataset** | `legal_training_data_batch/train.jsonl` & `val.jsonl` |
| **Records** | 5,509 (4,956 train / 553 val) |
| **Task** | Instruction-following: case facts → judgement |
| **Base Model** | `gpt2` (upgradable to `gpt2-medium`) |

> 💡 **GPU recommended** for fine-tuning. CPU works for inference only.

In [25]:
!pip install --upgrade transformers datasets accelerate peft -q

In [26]:
import os, json, math, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments,
    DataCollatorForLanguageModeling,
)

print(f'PyTorch: {torch.__version__}')

# ── Device detection (works on Kaggle T4, M1 MPS, and CPU) ──
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Device : {DEVICE}')
print(f'CUDA   : {torch.cuda.is_available()}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


PyTorch: 2.9.0+cu126
Device : cuda
CUDA   : True
GPU    : Tesla P100-PCIE-16GB


In [27]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [28]:
# ── Paths (Kaggle) ──
from pathlib import Path

# Your Kaggle dataset path — update DATASET_SLUG if needed
DATASET_SLUG     = 'yashaskapoor/legal-dataset-india'
DATA_DIR         = Path(f'/kaggle/input/{DATASET_SLUG}/legal_training_data_batch')
MODEL_OUTPUT_DIR = Path('/kaggle/working/legal_model')
print('Running on Kaggle')

# ── Use split files (common_judgement / different_judgement) ──
TRAIN_FILE = '/kaggle/input/datasets/yashaskapoor/legal-dataset-india/legal_training_data_batch/train_split.jsonl'
VAL_FILE   = '/kaggle/input/datasets/yashaskapoor/legal-dataset-india/legal_training_data_batch/val_split.jsonl'


# ── Hyperparameters ──
BASE_MODEL_NAME = 'gpt2-medium'   # Use 'gpt2-medium' for higher quality on T4
MAX_SEQ_LEN     = 256
BATCH_SIZE      = 4        # T4 can handle 8; reduce to 4 on M1; 1 on CPU
GRAD_ACCUM      = 4        # Effective batch = 16
NUM_EPOCHS      = 5
LEARNING_RATE   = 5e-5
WARMUP_STEPS    = 100
SAVE_STEPS      = 500
EVAL_STEPS      = 500
print(f'Output dir: {MODEL_OUTPUT_DIR}')


Running on Kaggle
Output dir: /kaggle/working/legal_model


In [29]:
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(TRAIN_FILE)
val_raw   = load_jsonl(VAL_FILE)
print(f'Train: {len(train_raw)} | Val: {len(val_raw)}')
s = train_raw[0]
print(f'Instruction: {s["instruction"]}')
print(f'Input[:150]: {s["input"][:150]}...')
print(f'Common[:150]    : {s["common_judgement"][:150]}...')
print(f'Different[:150] : {s["different_judgement"][:150]}...')

Train: 4956 | Val: 553
Instruction: Summarize this case related to IPC Section 294 in 2-3 sentences.
Input[:150]: Facts: Pawan Kumar vs State Of Haryana And Anr on 7 May, 1996 , K.S. Paripoornan...
Common[:150]    : PARIPOORNAN, K.S.(J) CITATION: 1996 SCC (4) 17 JT 1996 (5) 155 1996 SCALE (4)480 ACT: HEADNOTE: JUDGMENT: Punchhi. Special leave granted....
Different[:150] : This appeal is directed against the judgment and decree of the Punjab and Haryan...


In [30]:
# ── Format records using Common / Different judgement fields ──

PROMPT_TEMPLATE = (
    '### Instruction:\n{instruction}\n\n'
    '### Facts:\n{facts}\n\n'
    '### Common Judgement:\n{common}\n\n'
    '### Different Judgement:\n{different}'
)

def format_record(rec):
    facts = rec.get('input', '')[:800]  # Keep within token budget
    return PROMPT_TEMPLATE.format(
        instruction=rec.get('instruction', '').strip(),
        facts=facts.strip(),
        common=rec.get('common_judgement', '').strip(),
        different=rec.get('different_judgement', '').strip(),
    )

train_texts = [format_record(r) for r in train_raw]
val_texts   = [format_record(r) for r in val_raw]

print(f'Formatted {len(train_texts)} train samples')
print('\nSample prompt (first 500 chars):')
print(train_texts[0][:1500])


Formatted 4956 train samples

Sample prompt (first 500 chars):
### Instruction:
Summarize this case related to IPC Section 294 in 2-3 sentences.

### Facts:
Facts: Pawan Kumar vs State Of Haryana And Anr on 7 May, 1996 , K.S. Paripoornan

### Common Judgement:
PARIPOORNAN, K.S.(J) CITATION: 1996 SCC (4) 17 JT 1996 (5) 155 1996 SCALE (4)480 ACT: HEADNOTE: JUDGMENT: Punchhi. Special leave granted.

### Different Judgement:
This appeal is directed against the judgment and decree of the Punjab and Haryan


In [31]:
# ── Load tokenizer & add pad token ──
tokenizer = GPT2Tokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

def tokenize(texts):
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        return_tensors=None,
    )

print('Tokenizing train set (may take a minute)...')
train_tok = tokenize(train_texts)
val_tok   = tokenize(val_texts)
print('Tokenization complete!')
print(f'Train input_ids shape: {len(train_tok["input_ids"])} x {len(train_tok["input_ids"][0])}')

Tokenizing train set (may take a minute)...
Tokenization complete!
Train input_ids shape: 4956 x 256


In [32]:
# ── Create HuggingFace Datasets ──
train_dataset = Dataset.from_dict({
    'input_ids':      train_tok['input_ids'],
    'attention_mask': train_tok['attention_mask'],
    'labels':         train_tok['input_ids'],   # for CLM, labels = input_ids
})
val_dataset = Dataset.from_dict({
    'input_ids':      val_tok['input_ids'],
    'attention_mask': val_tok['attention_mask'],
    'labels':         val_tok['input_ids'],
})

train_dataset = train_dataset.with_format('torch')
val_dataset   = val_dataset.with_format('torch')

print(f'Train dataset: {train_dataset}')
print(f'Val dataset  : {val_dataset}')

Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4956
})
Val dataset  : Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 553
})


In [33]:
# ── Load GPT-2 model ──
model = GPT2LMHeadModel.from_pretrained(BASE_MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params   : {total_params:,}')
print(f'Trainable params: {trainable:,}')

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Total params   : 354,823,168
Trainable params: 354,823,168


In [34]:
# 1. Define the training arguments 
training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",    
    save_strategy="epoch",
    fp16=(DEVICE == 'cuda'), 
    logging_steps=50,
    report_to="none" 
)

# 2. Set up the Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   
)

# 3. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args, 
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer, # <-- This is the magic fix!
)
print('Trainer ready!')

Trainer ready!


In [35]:
import gc
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU memory free: 7.59 GB


In [36]:
# ── Train the model ──
# WARNING: This takes time.
# GPU (T4/V100): ~15-30 min for 3 epochs
# CPU: Very slow — reduce NUM_EPOCHS to 1 and BATCH_SIZE to 1

print('Starting training...')
trainer.train()
print('Training complete!')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting training...


Epoch,Training Loss,Validation Loss
1,1.773590,1.596161
2,1.381229,1.284246
3,1.209179,1.093281
4,1.040515,0.989135
5,0.970965,0.954560


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:664] . unexpected pos 2467026368 vs 2467026260

In [37]:
# ── Evaluate on validation set ──
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])
print(f'Eval Loss   : {eval_results["eval_loss"]:.4f}')
print(f'Perplexity  : {perplexity:.2f}')
# Lower perplexity = better model. Good range: 20-50 for this domain.

Eval Loss   : 0.9546
Perplexity  : 2.60


## Inference

After training and saving, run the cells below to generate structured justifications.
The model will produce **Common Judgement** and **Different Judgement** sections separately.


In [39]:
# Override the saved generation_config max_length
model.generation_config.max_length = 1024


In [41]:
from transformers import GenerationConfig

# Override saved generation_config BEFORE creating pipeline
model.generation_config = GenerationConfig(
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    pad_token_id=tokenizer.eos_token_id,
)

generator = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=0 if DEVICE == 'cuda' else -1,
)

def _generate(prompt):
    outputs = generator(prompt)   # params come from generation_config, no conflicts
    return outputs[0]['generated_text'][len(prompt):].strip()


def generate_justification(section_code, law_type='IPC'):
    # ── Call 1: Common Judgement ──
    common_prompt = (
        f'### Instruction:\nDescribe the common judgement for {law_type} Section {section_code}.\n\n'
        f'### Facts:\n\n'
        f'### Common Judgement:\n'
    )
    common = _generate(common_prompt)

    # ── Call 2: Different Judgement ──
    diff_prompt = (
        f'### Instruction:\nDescribe notable or different judgements for {law_type} Section {section_code}.\n\n'
        f'### Facts:\n\n'
        f'### Different Judgement:\n'
    )
    different = _generate(diff_prompt)

    return {'common': common, 'different': different}


# ── Test inference on sample sections ──
test_sections = [
    ('302', 'IPC'),
    ('376', 'IPC'),
    ('64',  'BNS'),
    ('103', 'BNS'),
]

for section, stype in test_sections:
    print(f'\n{"="*60}')
    print(f'{stype} Section {section}')
    print('='*60)
    result = generate_justification(section, stype)
    print(f'COMMON:\n{result["common"][:300]}')
    print(f'\nDIFFERENT:\n{result["different"][:300]}')


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



IPC Section 302


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COMMON:
Courts have consistently applied this section in cases matching the statutory criteria. 28. In State of Gujarat v. Manoranjan, (2004) 4 SCC 775, a three-Judge Bench consisting Of Justice Ajoy Mehta, J. and Mr Harshvardhan Singh Thakur observed as under:- "7.....it is clear that an order directing ar

DIFFERENT:
Dictum No.: 304/2013 vs State Of Haryana And Anr on 5 January, 2014, Advocate Mr.(S) Aashish Yadav Agee Counsel For the Petitioner :- Pradeep Kumar Sharma Court Nos./1 & 2 HC Ranjit Singh JUDGMENT 1.) This petition under Article 226 of Bhartiya Nagrik Suraksha Sanhita (BNSS), 2023 has been filed by 

IPC Section 376


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COMMON:
In view of above discussion, we are unable to agree with the finding made by learned Single Judge and as such he is not justified in quashing this criminal proceeding against petitioner-accused No.-2/A1 on his own ground that there was no proper investigation conducted or sufficient evidence collect

DIFFERENT:
Facts:- facts of the case, it is to be seen how far consent given by either party constituting triable joint venture can give rise under s 47 of the Code t o sustain a claim against an unincorporated body corporate as that term is defined in section 77(1)of The Corporations Act &c; and even then the

BNS Section 64


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COMMON:
Accordingly, we set aside and quash the impugned criminal proceedings." 528/11 SCC (3) at p 876 of 11 The Hon'ble Supreme Court observed as under:- "In our view no valid charge could be preferred against respondent No 2 on the ground that he is a police officer whereas complainant Nos 1 to 3 are pub

DIFFERENT:
Facts:- prosecution case, however the learned Single Judge found that in absence of any special material which would make it an indispensable piece to establish the guilt beyond reasonable doubt and also without revealing certain incriminating circumstances, prima facie established the offence punis

BNS Section 103


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COMMON:
In view of above discussion, this Court is not persuaded that there was no factory accident resulting in death on the body part(s) of Petitioner No 1 and also taking into consideration the fact (i). That said – even if present petitioners are found guilty under Sections 323/338 IPC then their convic

DIFFERENT:
Facts are hazy and the entire evidence/material is not before this Court, it cannot be said that no offence has been made out against petitioner in terms of the FIR as per CrlM No 298 /2024." 14 16. As regards charge under section 506 IPC - "production by circumstantial means" (Section 34(1)(c) rt m


In [42]:
import shutil

# Delete checkpoint folders, keep only the final model files
for item in Path(MODEL_OUTPUT_DIR).iterdir():
    if item.is_dir() and item.name.startswith('checkpoint'):
        shutil.rmtree(item)
        print(f'Deleted {item.name}')

print('Done! Remaining files:', [f.name for f in MODEL_OUTPUT_DIR.iterdir()])


Deleted checkpoint-1550
Deleted checkpoint-930
Deleted checkpoint-1240
Deleted checkpoint-620
Deleted checkpoint-310
Done! Remaining files: ['config.json']


In [ ]:
# ── Save the fine-tuned model ──
model.save_pretrained(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))
print(f'Model saved to: {MODEL_OUTPUT_DIR}')
print('Files:', [f.name for f in MODEL_OUTPUT_DIR.iterdir()])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/legal_model
Files: ['model.safetensors', 'config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
# ── Reload in FP16 and re-save ──
import torch
from transformers import GPT2LMHeadModel

model_fp16 = GPT2LMHeadModel.from_pretrained(
    str(MODEL_OUTPUT_DIR),
    torch_dtype=torch.float16   # load directly in FP16
)
model_fp16.save_pretrained(str(MODEL_OUTPUT_DIR))
print('Saved in FP16')

# Verify new size
for f in Path(MODEL_OUTPUT_DIR).iterdir():
    size_mb = os.path.getsize(f) / 1e6
    print(f'{size_mb:.1f} MB  {f.name}')


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved in FP16
709.7 MB  model.safetensors
0.0 MB  config.json
0.0 MB  generation_config.json
3.6 MB  tokenizer.json
0.0 MB  tokenizer_config.json


In [ ]:
# ── Zip the model for download from Kaggle Output tab ──
import shutil

zip_path = '/kaggle/working/legal_model_export'
shutil.make_archive(zip_path, 'zip', str(MODEL_OUTPUT_DIR))
print(f'Model zipped to: {zip_path}.zip')
print('Download it from the Kaggle Output tab on the right panel.')


Model zipped to: /kaggle/working/legal_model_export.zip
Download it from the Kaggle Output tab on the right panel.
